Metrics (2)

- avg_sen_len — mean sentence length (tokens)
- std_sen_len — std of sentence length

In [ ]:
import sys
import importlib
import spacy

importlib.invalidate_caches() # see packages installed after the kernel started
# download only if missing (uses the kernel's own python)
if not spacy.util.is_package("en_core_web_trf"):
    !{sys.executable} -m spacy download en_core_web_trf
print("en_core_web_trf installed:", spacy.util.is_package("en_core_web_trf"))

In [ ]:
import math, spacy
from spacy.tokens import Doc

# en_core_web_trf - same parser as mean-mdl.ipynb and kazmi-syntactic.ipynb, so sentences and POS tags come from the same parse
# only pos_ (tagger + attribute_ruler) and sents (parser) are needed, so ner and lemmatizer are disabled
spacy.prefer_gpu() # uses the Mac GPU (MPS) if available
nlp = spacy.load("en_core_web_trf", disable=["ner", "lemmatizer"])
print(f"Language model: {nlp.meta['lang']}_{nlp.meta['name']}")
print(f"Version: {nlp.meta['version']}")
print(f"Pipe names: {nlp.pipe_names}")

def calculate_kazmi_raw(text):
    #text can be a raw string or an already parsed doc (from nlp.pipe), so the paragraph is never parsed twice
    doc = text if isinstance(text, Doc) else nlp(text or "")
    sen_lens = []
    for sent in doc.sents:
        length = 0
        for token in sent:
            if token.pos_ != "SPACE":
                length += 1
        sen_lens.append(length)

    if not sen_lens:
        return {"avg_sen_len": f"{0:.5f}", "std_sen_len": f"{0:.5f}"}

    n = len(sen_lens)
    avg_sen_len = sum(sen_lens) / n

    variance_sum = 0
    for length in sen_lens:
        variance_sum += (length - avg_sen_len) ** 2
    std_sen_len = math.sqrt(variance_sum / n)

    return {
        "avg_sen_len": f"{avg_sen_len:.5f}",
        "std_sen_len": f"{std_sen_len:.5f}",
    }

In [ ]:
from pathlib import Path
import csv

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/raw-csv to read and data/<LEVEL>/features/ to write
raw_csv_path = Path(f"../../data/{LEVEL}/raw-csv")
# old values (en_core_web_sm) are in archive-pooled-sm-2026-11-11/kazmi-raw-csv
out_csv_path = Path(f"../../data/{LEVEL}/features/kazmi-raw-csv")
out_csv_path.mkdir(parents=True, exist_ok=True)

for csv_file in sorted(raw_csv_path.glob("*.csv")):
    out_csv_file = out_csv_path / csv_file.name
    if out_csv_file.exists() and out_csv_file.stat().st_size > 0: # Skip if already exist
       continue
    print(f"Working on file: {csv_file.name}")
    with csv_file.open("r",encoding="utf-8") as cf:
       csv_reader = csv.DictReader(cf)
       column_names = list(csv_reader.fieldnames)
       column_names = column_names[:-1] + ["avg_sen_len", "std_sen_len"] + [column_names[-1]]
       rows = list(csv_reader)
    # write to a temp file and rename at the end, so a half written file is never skipped on restart
    tmp_csv_file = out_csv_file.with_suffix(".tmp")
    with tmp_csv_file.open("w",encoding="utf-8") as of:
       csv_writer = csv.DictWriter(of,fieldnames=column_names)
       csv_writer.writeheader()
       # nlp.pipe parses texts in batches - much faster than calling nlp(text) for every row
       for row, doc in zip(rows, nlp.pipe([row["text"] or "" for row in rows], batch_size=64)):
          raw_feats = calculate_kazmi_raw(doc)
          row["avg_sen_len"] = raw_feats["avg_sen_len"]
          row["std_sen_len"] = raw_feats["std_sen_len"]
          csv_writer.writerow(row)
    tmp_csv_file.replace(out_csv_file)
    #break - During demo we can uncomment and providea demo path

In [8]:
# TEST KAZMI RAW CODE

text = "Use a dehydrator to dry them if you have one. Set your dehydrator to 150 F degrees. Lay the almonds in a single layer on the dehydrator trays. Place the almonds in the dehydrator for 12 to 24 hours, or until crispy."
print(calculate_kazmi_raw(text))
text = "Beyoncé and Rihanna are tied for the second - most #1 entries on the Billboard Hot 100 chart, with 5 each."
print(calculate_kazmi_raw(text))

{'avg_sen_len': '11.75000', 'std_sen_len': '2.86138'}
{'avg_sen_len': '24.00000', 'std_sen_len': '0.00000'}
